In [0]:
use catalog identifier(:catalog);

In [0]:
CREATE OR REPLACE TABLE silver.stations AS
SELECT
  upper(trim(station_id)) as station_id,
  trim(station_name) AS station_name,
  initcap(trim(city)) AS city,
  upper(trim(site_type)) AS site_type,
  cast(opened_on AS DATE) AS opened_on
FROM bronze.stations;

CREATE OR REPLACE TABLE silver.tariffs AS
SELECT
  upper(trim(connector_type)) AS connector_type,
  upper(trim(time_band)) AS time_band,
  cast(rate_per_kwh AS STRING) AS rate_per_kwh,
  cast(idle_fee_per_min AS DECIMAL(10,2)) AS idle_fee_per_min,
  cast(free_idle_minutes AS INT) AS free_idle_minutes
FROM bronze.tariffs;



In [0]:
CREATE TABLE IF NOT EXISTS silver.chargers (
  charger_id STRING,
  station_id STRING,
  connector_type STRING,
  max_kw DECIMAL(6,1),
  is_active BOOLEAN,
  commissioned_on DATE,
  last_changed_snapshot_date DATE
);

MERGE INTO silver.chargers AS t
USING (
  SELECT
    nullif(upper(trim(charger_id)), '')   AS charger_id,
    nullif(upper(trim(station_id)), '')   AS station_id,
    upper(trim(connector_type)) AS connector_type,
    cast(max_kw AS DECIMAL(6,1)) AS max_kw,
    cast(is_active AS BOOLEAN) AS is_active,
    cast(commissioned_on AS DATE)  AS commissioned_on,
    cast(snapshot_date AS DATE)  AS last_changed_snapshot_date
  FROM (
    SELECT charger_id, station_id, connector.type as connector_type,connector.max_kw as max_kw, is_active, commissioned_on, snapshot_date,
           row_number() OVER (
             PARTITION BY upper(trim(charger_id))
             ORDER BY snapshot_date DESC
           ) AS rn
    FROM bronze.chargers
  )
  WHERE rn = 1
) AS s
ON t.charger_id = s.charger_id

WHEN MATCHED AND NOT (
      t.station_id       <=> s.station_id
  AND t.connector_type  <=> s.connector_type
  AND t.max_kw         <=> s.max_kw
  AND t.is_active       <=> s.is_active
  AND t.commissioned_on <=> s.commissioned_on
)
THEN UPDATE SET
  station_id                 = s.station_id,
  connector_type              = s.connector_type,
  max_kw                      = s.max_kw,
  is_active                   = s.is_active,
  commissioned_on             = s.commissioned_on,
  last_changed_snapshot_date = s.last_changed_snapshot_date
WHEN NOT MATCHED THEN INSERT (
  charger_id, station_id, connector_type, max_kw, is_active, commissioned_on, last_changed_snapshot_date
) VALUES (
  s.charger_id, s.station_id, s.connector_type, s.max_kw, s.is_active, s.commissioned_on, s.last_changed_snapshot_date
);

ALTER TABLE silver.chargers SET TBLPROPERTIES (delta.enableChangeDataFeed = true);


In [0]:
CREATE OR REPLACE TABLE silver.members AS
WITH standardized AS (

  SELECT DISTINCT
    nullif(upper(trim(member_id)), '') AS member_id,
    trim(full_name)AS full_name,
    lower(trim(email)) AS email,
    nullif(right(regexp_replace(phone, '[^0-9]', ''), 10), '') AS phone,
    initcap(trim(city)) AS city,
    upper(trim(plan)) AS plan,
    coalesce(try_to_date(joined_on, 'yyyy-MM-dd'),
             try_to_date(joined_on, 'dd/MM/yyyy')) AS joined_on,
    CASE
      WHEN change_effective_date IS NULL OR trim(change_effective_date) = ''
        THEN DATE '2026-04-01'
      ELSE coalesce(try_to_date(change_effective_date, 'yyyy-MM-dd'),
                    try_to_date(change_effective_date, 'dd/MM/yyyy'))
    END AS effective_date
  FROM bronze.members
  WHERE nullif(upper(trim(member_id)), '') IS NOT NULL
),

ordered AS (
  SELECT
    member_id, full_name, email, phone, city, plan, joined_on, effective_date,
    ROW_NUMBER() OVER (PARTITION BY member_id ORDER BY effective_date) AS rn,
    LAG(full_name) OVER (PARTITION BY member_id ORDER BY effective_date) AS prev_full_name,
    LAG(email)     OVER (PARTITION BY member_id ORDER BY effective_date) AS prev_email,
    LAG(phone)     OVER (PARTITION BY member_id ORDER BY effective_date) AS prev_phone,
    LAG(city)      OVER (PARTITION BY member_id ORDER BY effective_date) AS prev_city,
    LAG(plan)      OVER (PARTITION BY member_id ORDER BY effective_date) AS prev_plan
  FROM standardized
),

flagged AS (
  SELECT
    member_id, full_name, email, phone, city, plan, joined_on, effective_date,
    CASE
      WHEN rn = 1 THEN 1
      WHEN NOT (plan <=> prev_plan)
        OR NOT (city <=> prev_city) THEN 1
      ELSE 0
    END AS is_version_start,
    
    CASE
      WHEN rn = 1 THEN 0
      WHEN (full_name <=> prev_full_name)
       AND (email     <=> prev_email)
       AND (phone     <=> prev_phone)
       AND (city      <=> prev_city)
       AND (plan      <=> prev_plan)           THEN 1
      ELSE 0
    END AS is_no_change
  FROM ordered
),

filtered AS (

  SELECT
    member_id, full_name, email, phone, city, plan, joined_on, effective_date,
    SUM(is_version_start) OVER (PARTITION BY member_id ORDER BY effective_date) AS version_num
  FROM flagged
  WHERE is_no_change = 0
),

version_rows AS (
 
  SELECT
    member_id,
    version_num,
    full_name, email, phone, city, plan, joined_on,
    MIN(effective_date) OVER (PARTITION BY member_id, version_num) AS effective_from
  FROM filtered
  QUALIFY ROW_NUMBER() OVER (PARTITION BY member_id, version_num ORDER BY effective_date DESC) = 1
)

SELECT
  member_id,
  full_name,
  email,
  phone,
  city,
  plan,
  FIRST_VALUE(joined_on) OVER (PARTITION BY member_id ORDER BY effective_from) AS joined_on,
  effective_from,
  LEAD(effective_from) OVER (PARTITION BY member_id ORDER BY effective_from) AS effective_to,
  LEAD(effective_from) OVER (PARTITION BY member_id ORDER BY effective_from) IS NULL AS is_current
FROM version_rows;